# WHOI Cable notebook designer

Create parts, assemble an anchored mooring, configure its forcing, export CABLE input and run the supplied solvers from this notebook.

**Start here:** use the **Python (whoi-cable)** kernel inside Linux/WSL and choose **Run → Run All Cells**. The interface appears below the application-code cell. The code is embedded, so this notebook does not need a separate Python module.

The form builder supports **2D single-anchor surface and subsurface moorings**. The raw CABLE input tab preserves other 2D input syntax. The supplied `cable-free` buoy patch applies to 2D surface moorings with Morison forcing.

If widgets are missing, install them into the notebook kernel: `%pip install ipywidgets numpy scipy matplotlib`. For JupyterLab running in another environment, install `jupyterlab_widgets` there too. The included updated `environment.yml` supplies the required packages.


In [4]:
# Usually leave these as None: the notebook finds its enclosing repository.
# Or point at your existing extracted copy of the attached WHOI Cable version.
CABLE_ROOT = '/home/smiro41p/whoi_cable'       # e.g. "/home/rob/whoi_cable"
PROJECTS_DIR = None     # None => <repository>/notebook_projects

# Uncomment only if your selected kernel is missing the GUI dependencies:
# %pip install ipywidgets numpy scipy matplotlib


In [5]:
"""WHOI Cable notebook designer, GPL-3.0-or-later.

Built for the supplied whoi_cable-master version. The solver is unchanged.
The form editor supports 2D, single-anchor surface/subsurface moorings.
"""
from __future__ import annotations

import ast
import asyncio
import copy
import datetime as dt
import html
import json
import math
import os
from pathlib import Path
import re
import shlex
import shutil
import signal
import sys

VERSION = "1.0.0"
KINDS = {"Materials": "Line / chain / rigid instrument", "Connectors": "Shackle / instrument / float", "Buoys": "Terminal buoy", "Anchors": "Anchor"}
HEADINGS = ("Problem Description", "Analysis Parameters", "Environment", "Buoys", "Anchors", "Materials", "Connectors", "Layout", "End")
ALIASES = {"mass": "m", "diam": "d", "height": "h", "SWL": "swl"}
STRINGS = {"comment", "category", "color"}
FIELDS = {
    "Materials": [("type", "Material law", "linear"), ("m", "Mass (kg/m)", 0.068), ("wet", "Wet weight (N/m)", -0.086), ("d", "Effective diameter (m)", 0.012), ("EA", "Axial stiffness EA (N)", 120000), ("EI", "Bending stiffness EI (N m²)", 0.1), ("GJ", "Torsional stiffness GJ (N m²)", 0.1), ("Cdn", "Normal drag Cdn", 1.5), ("Cdt", "Tangential drag Cdt", 0.01), ("Cmn", "Normal inertia Cmn", None), ("Cmt", "Tangential inertia Cmt", None), ("amn", "Normal added mass (kg/m)", None), ("amt", "Tangential added mass (kg/m)", None), ("swl", "Safe working load (N)", None), ("yield", "Yield / breaking load (N)", None), ("length", "Fixed part length (m)", None)],
    "Connectors": [("m", "Mass in air (kg)", 0.17), ("wet", "Net wet weight (N)", 1.45), ("d", "Effective diameter (m)", 0.04), ("Cdn", "Normal drag Cdn", 1.0), ("Cdt", "Tangential drag Cdt", 0.0), ("am", "Added mass (kg)", None), ("length", "Body length (m)", None)],
    "Buoys": [("type", "Shape", "axisymmetric"), ("m", "Mass in air (kg)", 18.5), ("d", "Diameter (m)", None), ("h", "Cylinder height (m)", None), ("diameters", "Axisymmetric profile: height, diameter (m)", [(0, 0.3), (0.25, 0.6), (0.41, 0.6), (0.42, 0.1), (1.15, 0.1)]), ("Cdn", "Normal drag Cdn", 0.8), ("Cdt", "Tangential drag Cdt", 0.8), ("Cdw", "Wind drag Cdw", 1.0), ("Sw", "Wind area override (m²)", None), ("am", "Added mass (kg)", None)],
    "Anchors": [("m", "Mass in air (kg)", 0.0), ("wet", "Net wet weight (N)", 0.0), ("d", "Effective diameter (m)", 0.0), ("Cdn", "Normal drag Cdn", 0.0), ("Cdt", "Tangential drag Cdt", 0.0), ("mu", "Seabed friction coefficient", 0.5)]
}
EXTRA_KEYS = {
    "Materials": {"T", "Te", "Tee", "E", "G", "J", "I", "nu", "id", "A", "w", "am", "Can", "Cat", "bn", "bt"},
    "Buoys": {"buoyancy", "draft", "cg", "Cl", "b"},
    "Connectors": set(), "Anchors": set()
}
DEFAULT_ANALYSIS = {
    "static-relaxation": 0.1, "static-iterations": 20000, "static-tolerance": 0.001,
    "relax-adapt-up": 1.02, "relax-adapt-down": 1.1, "static-outer-iterations": 1000,
    "static-initial-guess": "catenary", "static-solution": "relaxation", "static-outer-relaxation": 0.98,
    "static-outer-tolerance": 0.01, "duration": 60.0, "ramp-time": 16.0, "time-step": 0.02,
    "dynamic-relaxation": 1.0, "dynamic-iterations": 15, "dynamic-tolerance": 1e-6
}


def number(value):
    """Evaluate only finite literal arithmetic, never Python code or variables."""
    if isinstance(value, (int, float)):
        result = float(value)
    else:
        tree = ast.parse(str(value).strip(), mode="eval")
        if len(list(ast.walk(tree))) > 80:
            raise ValueError("Numeric expression is too long")
        def walk(n):
            if isinstance(n, ast.Constant) and type(n.value) in (int, float):
                return float(n.value)
            if isinstance(n, ast.UnaryOp) and isinstance(n.op, (ast.UAdd, ast.USub)):
                v = walk(n.operand)
                return v if isinstance(n.op, ast.UAdd) else -v
            if isinstance(n, ast.BinOp):
                a, b = walk(n.left), walk(n.right)
                if isinstance(n.op, ast.Add): return a + b
                if isinstance(n.op, ast.Sub): return a - b
                if isinstance(n.op, ast.Mult): return a * b
                if isinstance(n.op, ast.Div): return a / b
            raise ValueError("Use a number or literal arithmetic (+, -, *, /)")
        result = walk(tree.body)
    if not math.isfinite(result):
        raise ValueError("Value must be finite")
    return result


def integer(value):
    n = number(value)
    if n != int(n): raise ValueError("Expected a whole number")
    return int(n)


def fmt(value):
    return f"{number(value):.12g}"


def quote(value):
    # Cable's lexer has no general escape-sequence decoder.
    s = str(value)
    if any(c in s for c in ['"', '\\', '\n', '\r', '\x00']):
        raise ValueError('Text cannot contain quotes, backslashes, or newlines')
    return '"' + s + '"'


def clean_comments(text):
    return re.sub(r'/\*.*?\*/|"[^"\n]*"', lambda m: m[0] if m[0].startswith('"') else "\n" * m[0].count("\n") + " ", text, flags=re.S)


def sections(text):
    clean = clean_comments(text)
    pattern = r"(?im)^\s*(" + "|".join(HEADINGS) + r")\s*$"
    matches = list(re.finditer(pattern, clean))
    return {m[1].lower(): clean[m.end(): matches[i+1].start() if i+1 < len(matches) else len(clean)] for i, m in enumerate(matches)}


def assignments(text):
    # Names/values may span lines; quoted strings are kept opaque.
    pattern = r'"[^"\n]*"|\b([A-Za-z_][\w-]*)\s*='
    matches = [m for m in re.finditer(pattern, text) if m[1]]
    return {ALIASES.get(m[1], m[1]): text[m.end():matches[i+1].start() if i+1 < len(matches) else len(text)].strip() for i, m in enumerate(matches)}


def tuples(value, width=2):
    rows = []
    rest = str(value)
    for m in re.finditer(r"\(([^()]*)\)", str(value)):
        row = [number(s) for s in m[1].split(",")]
        if len(row) != width: raise ValueError(f"Expected {width} values per tuple")
        rows.append(row)
    rest = re.sub(r"\([^()]*\)", "", rest).strip(" \n\t,")
    if rest or not rows: raise ValueError(f"Expected (value, ...) tuples, got {value!r}")
    return rows


def rows_text(rows):
    return "\n".join(", ".join(fmt(x) for x in row) for row in rows)


def parse_rows(text, width=2):
    if "(" in text: return tuples(text, width)
    rows = []
    for line in text.splitlines():
        if not line.strip(): continue
        row = [number(x) for x in re.split(r"[,;\s]+", line.strip())]
        if len(row) != width: raise ValueError(f"Each row needs {width} numbers")
        rows.append(row)
    if not rows: raise ValueError("Enter at least one row")
    return rows


def read_parts(text, source=""):
    catalog = {k: {} for k in KINDS}
    for kind in KINDS:
        body = sections(text).get(kind.lower(), "")
        # Object declarations start a line; field continuations must not match.
        start = re.compile(r'(?m)^\s*("[^"\n]+"|[A-Za-z_][\w/.:]*)(?=\s|$)(?!\s*=)')
        marks = list(start.finditer(body))
        for i, m in enumerate(marks):
            name = m[1].strip('"')
            raw = assignments(body[m.end():marks[i+1].start() if i+1 < len(marks) else len(body)])
            props = {}
            for key, val in raw.items():
                if key in STRINGS: props[key] = val.strip('"')
                elif key == "diameters": props[key] = tuples(val)
                elif key == "type": props[key] = val
                else:
                    try: props[key] = number(val)
                    except (ValueError, SyntaxError): props[key] = val
            if props:
                catalog[kind][name] = {"properties": props, "source": source}
    return catalog


def validate_part(kind, name, props):
    if kind not in KINDS: raise ValueError("Unknown part kind")
    if not name.strip(): raise ValueError("Part needs a name")
    quote(name)
    allowed = {k for k, _, _ in FIELDS[kind]} | EXTRA_KEYS[kind] | STRINGS
    unknown = set(props) - allowed
    if unknown: raise ValueError("Unsupported part fields: " + ", ".join(sorted(unknown)))
    for key, value in props.items():
        if key in STRINGS: quote(value)
        elif key == "type":
            options = ("linear", "nonlinear") if kind == "Materials" else ("axisymmetric", "sphere", "cylinder")
            if value not in options: raise ValueError(f"Unsupported {kind} type: {value}")
        elif key == "diameters":
            if len(value) < 2 or any(len(r) != 2 for r in value): raise ValueError("Buoy profile needs at least two height,diameter pairs")
            if number(value[0][0]) != 0: raise ValueError("Buoy profile must start at height 0")
            if any(number(r[1]) < 0 for r in value): raise ValueError("Profile diameters cannot be negative")
            if any(number(b[0]) <= number(a[0]) for a, b in zip(value, value[1:])): raise ValueError("Profile heights must increase strictly")
            if not any(number(r[1]) > 0 for r in value): raise ValueError("Buoy needs nonzero volume")
        elif key in {"T", "Te", "Tee"}:
            if not str(value).strip() or re.search(r'[{}\n\r;=#]', str(value)): raise ValueError("Invalid constitutive expression")
        else:
            n = number(value)
            if key not in {"wet", "draft", "cg", "bn", "bt"} and n < 0: raise ValueError(f"{key} cannot be negative")
    for required in {"Materials": ["m", "wet", "d"], "Connectors": ["m", "wet", "d"], "Buoys": ["m", "type"], "Anchors": []}[kind]:
        if required not in props: raise ValueError(f"Missing {required}")
    if kind == "Materials":
        if props.get("type", "linear") == "nonlinear":
            if not all(k in props for k in ("T", "Te", "Tee")): raise ValueError("Nonlinear material requires T, Te and Tee")
        elif number(props.get("EA", 0)) <= 0: raise ValueError("EA must be positive")
        if number(props.get("d", 0)) <= 0: raise ValueError("Line diameter must be positive")
    if kind == "Buoys":
        shape = props.get("type")
        if shape == "axisymmetric" and "diameters" not in props: raise ValueError("Axisymmetric buoy needs a diameter profile")
        if shape in ("sphere", "cylinder") and number(props.get("d", 0)) <= 0: raise ValueError("Buoy diameter must be positive")
        if shape == "cylinder" and number(props.get("h", 0)) <= 0: raise ValueError("Cylinder height must be positive")


def buoy_volume(props):
    shape = props.get("type")
    if shape == "sphere": return math.pi * number(props["d"])**3 / 6
    if shape == "cylinder": return math.pi * number(props["d"])**2 * number(props["h"]) / 4
    if shape == "axisymmetric":
        return sum(math.pi*(number(b[0])-number(a[0]))*(number(a[1])**2+number(a[1])*number(b[1])+number(b[1])**2)/12 for a,b in zip(props["diameters"],props["diameters"][1:]))
    return 0.0


def atomic_write(path, text):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(text, encoding="utf-8")
    tmp.replace(path)


def discover_root(start=None):
    p = Path(start or Path.cwd()).expanduser().resolve()
    candidates = [p, *p.parents]
    for base in list(candidates):
        candidates += [base / "whoi_cable", base / "whoi_cable-master", base / "cable-master"]
    return next((p for p in candidates if (p / "model/parser.y").is_file() and (p / "scripts/build.sh").is_file()), None)


def default_project():
    return {"schema_version": 1, "name": "my_mooring", "title": "New surface mooring", "type": "surface",
            "anchor": "clump", "buoy": "slb600", "segments": [{"material": "polypro_12mm", "length": 50.0, "nodes": 51, "connector": None, "attachments": []}],
            "analysis": copy.deepcopy(DEFAULT_ANALYSIS),
            "environment": {"depth": 40., "rho": 1027., "gravity": 9.81, "x-wind": 0., "bottom-stiffness": 100., "bottom-damping": 1., "forcing-method": "morison", "input-type": "regular"},
            "current": [[0., 0.5], [40., 0.5]], "wave": [2., 8., 0.], "modulation": "",
            "run": {"solver": "cable-free", "static_only": False, "auto": False, "sample": 0.1, "snap_dt": 0.2}}


def reference_parts():
    return read_parts('''Buoys
slb600 type = axisymmetric
 diameters = (0,0.3) (0.25,0.6) (0.41,0.6) (0.42,0.1) (1.15,0.1)
 m = 18.5 Cdn = 0.8 Cdt = 0.8 Cdw = 1
 comment = "Approximate hull from supplied slb600_waves case"
Anchors
clump color = red
Materials
polypro_12mm EA = 120000 EI = 0.1 GJ = 0.1 m = 0.068 wet = -0.086 d = 0.012 Cdt = 0.01 Cdn = 1.5 Cmt = 1.1 Cmn = 2
chain_8mm EA = 24000000 EI = 10 GJ = 0.1 m = 1.4 wet = 11.9 d = 0.031 Cdt = 0.01 Cdn = 0.5 Cmt = 1.1 Cmn = 2
Connectors
shackle_10mm d = 0.04 m = 0.17 wet = 1.45 Cdn = 1 Cdt = 0
End
''', "Supplied example; approximate properties")


def load_catalog(root):
    catalog = reference_parts()
    if root:
        for p in sorted((Path(root) / "gui/database").glob("*.db")):
            if p.name == "current.db": continue
            data = read_parts(p.read_text(encoding="utf-8", errors="replace"), f"gui/database/{p.name}")
            for k in KINDS: catalog[k].update(data[k])
        # The supplied case definitions take precedence over generic GUI defaults.
        for p in sorted((Path(root) / "cases").glob("*.cbl")):
            data = read_parts(p.read_text(encoding="utf-8", errors="replace"), f"cases/{p.name}")
            for k in KINDS: catalog[k].update(data[k])
    return catalog


def used_parts(project, catalog):
    refs = {"Buoys": {project["buoy"]}, "Anchors": {project["anchor"]}, "Materials": set(), "Connectors": set()}
    for s in project["segments"]:
        refs["Materials"].add(s["material"])
        if s.get("connector"): refs["Connectors"].add(s["connector"])
        refs["Connectors"].update(a["part"] for a in s.get("attachments", []))
    result = {k: {} for k in KINDS}
    for kind, names in refs.items():
        for name in sorted(names):
            if name not in catalog[kind]: raise ValueError(f"Missing {kind} part: {name}")
            result[kind][name] = copy.deepcopy(catalog[kind][name])
    return result


def validate_project(p, catalog):
    if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_-]*", p["name"]): raise ValueError("Case name: use letters, numbers, _ or -; start with a letter/number")
    quote(p["title"])
    if p["type"] not in ("surface", "subsurface"): raise ValueError("Form mode supports surface and subsurface moorings")
    if not p["segments"]: raise ValueError("Add at least one line segment")
    if len(p["segments"]) > 200: raise ValueError("Maximum 200 segments")
    ref = used_parts(p, catalog)
    for kind in KINDS:
        for name, entry in ref[kind].items(): validate_part(kind, name, entry["properties"])
    for i, s in enumerate(p["segments"], 1):
        if number(s["length"]) <= 0: raise ValueError(f"Segment {i}: length must be positive")
        if integer(s["nodes"]) < 2: raise ValueError(f"Segment {i}: at least 2 nodes required")
        for a in s.get("attachments", []):
            if not a["nodes"] or any(integer(n) < 1 or integer(n) > integer(s["nodes"]) for n in a["nodes"]): raise ValueError(f"Segment {i}: attachment nodes must be within 1..{s['nodes']}")
    e, a = p["environment"], p["analysis"]
    for key in ("depth", "rho", "gravity"):
        if number(e[key]) <= 0: raise ValueError(f"{key} must be positive")
    for key in ("bottom-stiffness", "bottom-damping"):
        if number(e.get(key, 0)) < 0: raise ValueError(f"{key} must be nonnegative")
    if e.get("forcing-method") != "morison" or e.get("input-type") != "regular": raise ValueError("Form mode uses regular waves with Morison forcing")
    for key in ("time-step", "duration", "static-tolerance", "static-outer-tolerance", "dynamic-tolerance", "static-relaxation", "dynamic-relaxation", "static-outer-relaxation", "relax-adapt-up", "relax-adapt-down"):
        if number(a[key]) <= 0: raise ValueError(f"{key} must be positive")
    for key in ("static-iterations", "static-outer-iterations", "dynamic-iterations"):
        if integer(a[key]) < 1: raise ValueError(f"{key} must be a positive integer")
    if number(a["ramp-time"]) < 0: raise ValueError("Ramp time cannot be negative")
    if a["static-initial-guess"] not in ("catenary", "shooting"): raise ValueError("Unsupported initial guess")
    if a["static-solution"] not in ("relaxation", "shooting", "catenary"): raise ValueError("Unsupported static solution")
    if number(a["time-step"]) > number(a["duration"]): raise ValueError("Time step exceeds duration")
    rows = p["current"]
    if not rows or any(len(r) != 2 for r in rows): raise ValueError("Current needs depth,speed pairs")
    if number(rows[0][0]) != 0: raise ValueError("Current profile must start at depth 0 (mean surface)")
    if any(number(b[0]) <= number(a[0]) for a,b in zip(rows, rows[1:])): raise ValueError("Current profile depths must increase strictly")
    if any(number(r[0]) < 0 or number(r[0]) > number(e["depth"]) for r in rows): raise ValueError("Current depths must lie within water depth")
    for r in rows: number(r[1])
    h, period, phase = map(number, p["wave"])
    if h < 0 or period <= 0: raise ValueError("Wave height must be nonnegative; period must be positive")
    mod = p.get("modulation", "").strip()
    if mod:
        if "(" in mod and re.match(r"^\(\s*[\d.+-]+\s*,", mod):
            pairs = tuples(mod)
            if len(pairs) < 2 or pairs[0][0] != 0 or any(b[0] <= a[0] for a,b in zip(pairs,pairs[1:])): raise ValueError("Current modulation needs increasing time,factor pairs starting at t=0")
        elif "t" not in mod or re.search(r'[{}\n\r;=#]', mod):
            raise ValueError("Current modulation needs time,factor pairs or an expression in t; a plain constant is unsafe in this version")
    for key in ("sample", "snap_dt"):
        if number(p["run"][key]) <= 0: raise ValueError(f"{key} must be positive")
    warnings = []
    total = sum(number(s["length"]) for s in p["segments"])
    if p["type"] == "surface" and total < number(e["depth"]): warnings.append("Line is shorter than water depth; buoy submergence or large stretch may result.")
    buoy = catalog["Buoys"][p["buoy"]]["properties"]
    if buoy_volume(buoy) * number(e["rho"]) <= number(buoy["m"]): warnings.append("Terminal buoy has no positive reserve buoyancy from the entered geometry.")
    if any("buoyancy" in x["properties"] and number(x["properties"]["buoyancy"]) > 0 for x in ref["Buoys"].values()): warnings.append("Buoyancy override is present in imported part; check consistency with its geometry.")
    if p["run"].get("auto"): warnings.append("-auto overrides the static solution strategy and can fail if the surface buoy starts submerged.")
    if number(a["ramp-time"]) >= number(a["duration"]) and not p["run"].get("static_only"): warnings.append("Run ends before the wave ramp has finished.")
    if number(a["duration"]) / number(a["time-step"]) > 2e6: warnings.append("More than two million time steps; consider a short convergence test first.")
    if number(rows[-1][0]) < number(e["depth"]): warnings.append("Current profile ends above seabed; Cable extends the endpoint value below the last pair.")
    return warnings


def part_text(kind, name, props):
    validate_part(kind, name, props)
    lines = ["    " + quote(name)]
    for key, value in props.items():
        if key in STRINGS: v = quote(value)
        elif key == "diameters": v = " ".join("(" + ", ".join(fmt(x) for x in r) + ")" for r in value)
        elif key == "type" or key in {"T", "Te", "Tee"}: v = str(value)
        else: v = fmt(value)
        lines.append(f"        {key} = {v}")
    return "\n".join(lines)


def export_cbl(p, catalog):
    validate_project(p, catalog)
    lines = ["/* Generated by WHOI Cable Notebook Designer " + VERSION + " */", "Problem Description", "    title = " + quote(p["title"]), "    type = " + p["type"], "", "Analysis Parameters"]
    for key, value in p["analysis"].items():
        lines.append(f"    {key} = {value if key in ('static-initial-guess','static-solution') else fmt(value)}")
    lines += ["", "Environment"]
    for key, value in p["environment"].items():
        lines.append(f"    {key} = {value if key in ('forcing-method','input-type') else fmt(value)}")
    if p.get("modulation", "").strip(): lines.append("    x-current-modulation = " + p["modulation"].strip())
    lines.append("    x-current = " + " ".join(f"({fmt(d)}, {fmt(u)})" for d,u in p["current"]))
    lines.append(f"    x-wave = ({fmt(number(p['wave'][0])/2)}, {fmt(p['wave'][1])}, {fmt(p['wave'][2])})")
    for kind, parts in used_parts(p, catalog).items():
        if parts:
            lines += ["", kind]
            for name, entry in parts.items(): lines.append(part_text(kind, name, entry["properties"]))
    lines += ["", "Layout", "    terminal = {", "        anchor = " + quote(p["anchor"]), "    }"]
    for s in p["segments"]:
        lines += ["    segment = {", "        length = " + fmt(s["length"]), "        material = " + quote(s["material"]), f"        nodes = ({integer(s['nodes'])}, 1.0)"]
        if s.get("attachments"):
            lines.append("        attachments = " + " ".join(quote(a["part"]) + " : (" + ", ".join(str(integer(n)) for n in a["nodes"]) + ")" for a in s["attachments"]))
        lines.append("    }")
        if s.get("connector"): lines.append("    connector = " + quote(s["connector"]))
    lines += ["    terminal = {", "        buoy = " + quote(p["buoy"]), "    }", "", "End", ""]
    return "\n".join(lines)


def import_cbl(text, name="imported_mooring"):
    """Strict import: reject syntax the forms cannot reproduce; raw mode stays available."""
    sec = sections(text)
    if not all(k in sec for k in ("problem description", "environment", "layout", "end")): raise ValueError("Not a complete Cable input file")
    desc = assignments(sec["problem description"])
    if set(desc) - {"title", "type"}: raise ValueError("Extra problem-description parameters need raw mode")
    p = default_project()
    p.update(name=re.sub(r"[^A-Za-z0-9_-]", "_", name).strip("_-") or "imported_mooring", title=desc.get("title", name).strip('"'), type=desc.get("type", "surface"))
    raw = assignments(sec.get("analysis parameters", ""))
    if set(raw) - set(DEFAULT_ANALYSIS): raise ValueError("Extra analysis parameters need raw mode: " + ", ".join(set(raw)-set(DEFAULT_ANALYSIS)))
    for k,v in raw.items(): p["analysis"][k] = v if k in ("static-initial-guess", "static-solution") else number(v)
    env = assignments(sec["environment"])
    allowed = set(p["environment"]) | {"x-current", "x-current-modulation", "x-wave"}
    if set(env) - allowed: raise ValueError("Extra environment parameters need raw mode: " + ", ".join(sorted(set(env)-allowed)))
    for k in p["environment"]:
        if k in env: p["environment"][k] = env[k] if k in ("forcing-method", "input-type") else number(env[k])
    current = env.get("x-current", "0")
    p["current"] = tuples(current) if "(" in current else [[0, number(current)], [p["environment"]["depth"], number(current)]]
    waves = tuples(env["x-wave"], 3) if "x-wave" in env else [[0,8,0]]
    if len(waves) != 1: raise ValueError("Multiple wave components need raw mode")
    p["wave"] = [2*waves[0][0], *waves[0][1:]]
    p["modulation"] = env.get("x-current-modulation", "")
    layout = sec["layout"]
    token = re.compile(r'(terminal|segment)\s*=\s*\{([^{}]*)\}|connector\s*=\s*("[^"\n]+"|[A-Za-z_][\w/.:]*)', re.S | re.I)
    marks = list(token.finditer(layout))
    if token.sub("", layout).strip(): raise ValueError("Branches, spools or complex connectors need raw mode")
    if len(marks) < 3 or marks[0][1] != "terminal" or marks[-1][1] != "terminal": raise ValueError("Expected two terminals and line segments")
    bottom, top = assignments(marks[0][2]), assignments(marks[-1][2])
    if set(bottom) != {"anchor"} or set(top) != {"buoy"}: raise ValueError("Terminal positions/forces or non-anchor moorings need raw mode")
    p["anchor"], p["buoy"] = bottom["anchor"].strip('"'), top["buoy"].strip('"')
    p["segments"] = []
    for m in marks[1:-1]:
        if m[1] == "segment":
            s = assignments(m[2])
            if set(s) - {"length", "material", "nodes", "attachments"}: raise ValueError("Nonstandard segment parameters need raw mode")
            dist = tuples(s["nodes"])
            if len(dist) != 1 or not math.isclose(dist[0][1], 1): raise ValueError("Nonuniform node distributions need raw mode")
            seg = {"material": s["material"].strip('"'), "length": number(s["length"]), "nodes": integer(dist[0][0]), "connector": None, "attachments": []}
            if s.get("attachments"):
                pattern = re.compile(r'("[^"\n]+"|[A-Za-z_][\w/.:]*)\s*:\s*\(([^()]*)\)')
                for a in pattern.finditer(s["attachments"]):
                    seg["attachments"].append({"part": a[1].strip('"'), "nodes": [integer(n) for n in re.split(r"[,\s]+", a[2].strip())]})
                if pattern.sub("", s["attachments"]).strip(" ,\n\t"): raise ValueError("Unsupported attachment syntax; use raw mode")
            p["segments"].append(seg)
        elif m[3]:
            if not p["segments"] or p["segments"][-1]["connector"]: raise ValueError("Multiple connectors at one joint need raw mode")
            p["segments"][-1]["connector"] = m[3].strip('"')
        else: raise ValueError("Extra terminal inside layout needs raw mode")
    return p, read_parts(text, "Imported .cbl")


def resolve_program(root, program, explicit=""):
    if explicit.strip():
        p = Path(explicit).expanduser().resolve()
        if not p.is_file() or not os.access(p, os.X_OK): raise FileNotFoundError(f"Not an executable: {p}")
        return str(p)
    variant = "free" if program == "cable-free" else "stock"
    filename = "cable" if program == "cable-free" else program
    candidates = []
    if root: candidates += [Path(root)/"build"/variant/"cli"/filename]
    candidates += [Path(sys.prefix)/"bin"/program]
    found = shutil.which(program)
    if found: candidates.append(Path(found))
    for p in candidates:
        if p.is_file() and os.access(p, os.X_OK): return str(p.resolve())
    raise FileNotFoundError(f"{program} not found. Activate the whoi-cable kernel and build using the Setup tab.")


def solver_command(executable, input_path, output_path, run, check=False):
    args = [str(executable), "-in", str(input_path), "-out", str(output_path), "-nocpp", "-quiet"]
    if check: return args + ["-debug"]
    args += ["-terminals", "-connectors", "-sample", fmt(run["sample"]), "-snap_dt", fmt(run["snap_dt"])]
    if run.get("static_only"): args.append("-static")
    if run.get("auto"): args.append("-auto")
    return args


class CableDesigner:
    """Interactive ipywidgets app; display(app.widget) inside Jupyter."""

    def __init__(self, cable_root=None, workspace=None):
        import ipywidgets as W
        from IPython.display import display, HTML
        self.W, self.display, self.HTML = W, display, HTML
        self.root = Path(cable_root).expanduser().resolve() if cable_root else discover_root()
        self.workspace = Path(workspace).expanduser().resolve() if workspace else (self.root or Path.cwd()) / "notebook_projects"
        self.catalog = load_catalog(self.root)
        self._overlay_catalog()
        self.project = default_project()
        template = self.root / "cases/slb600_waves.cbl" if self.root else None
        if template and template.is_file():
            self.project, _ = import_cbl(template.read_text(), "my_mooring")
        self.process = None
        self.task = None
        self.last_run = None
        self.cancel_requested = False
        self.raw_origin = None
        self._busy = False
        self._loading = False
        self._part_baseline = {}
        self.status = W.HTML()
        self._build_setup()
        self._build_parts()
        self._build_mooring()
        self._build_forcing()
        self._build_run()
        self._build_input()
        self.tabs = W.Tab(children=[self.setup_tab, self.parts_tab, self.mooring_tab, self.forcing_tab, self.run_tab, self.input_tab])
        for i,title in enumerate(("Setup & projects", "Parts", "Mooring", "Forcing & solver", "Run & results", "CABLE input")): self.tabs.set_title(i,title)
        self.widget = W.VBox([W.HTML('''<style>.cable-source textarea{font-family:monospace;font-size:12px;line-height:1.5}.cable-designer .widget-button{border-radius:5px}.cable-designer .widget-tab-contents{padding:16px}</style><div style="background:#10354b;color:white;padding:22px 26px;border-radius:12px;margin-bottom:12px"><div style="font-size:12px;letter-spacing:2px;color:#91d8e8">WHOI CABLE · NOTEBOOK DESIGNER</div><h2 style="margin:7px 0;color:white">Build your next mooring</h2><div>Parts → layout → forcing → review → run. Files stay on the notebook server.</div></div>'''), self.tabs, self.status], layout=W.Layout(width="100%", max_width="1200px"))
        self.widget.add_class("cable-designer")
        self._load_widgets()
        self._part_new()
        self._refresh_project_list()
        self._diagnostics()
        self._notice("Ready. Start from the loaded example, or create a new project. Parts and project changes are saved only when you click Save.")

    def _overlay_catalog(self):
        p = self.workspace / "parts.json"
        if p.is_file():
            data = json.loads(p.read_text())
            if data.get("schema_version") != 1: raise ValueError("Unsupported parts catalog version")
            for k in KINDS: self.catalog[k].update(data["parts"].get(k, {}))

    def _notice(self, message, error=False):
        color = "#a32c29" if error else "#125970"
        self.status.value = f'<div style="padding:12px;border-left:4px solid {color};background:#f4f8fa;margin-top:10px;color:{color}">{html.escape(str(message))}</div>'

    def _guard(self, fn):
        def cb(_=None):
            try: fn()
            except Exception as e: self._notice(f"{type(e).__name__}: {e}", True)
        return cb

    def _button(self, title, fn, primary=False, icon=""):
        b = self.W.Button(description=title, button_style="info" if primary else "", icon=icon, layout=self.W.Layout(width="auto", min_width="110px"))
        b.on_click(self._guard(fn))
        return b

    def _label(self, label, widget):
        widget.layout.width = "100%"
        return self.W.VBox([self.W.HTML(f'<span style="font-weight:600;font-size:12px">{html.escape(label)}</span>'), widget])

    def _grid(self, items):
        return self.W.GridBox(items, layout=self.W.Layout(grid_template_columns="repeat(auto-fit, minmax(240px, 1fr))", grid_gap="12px", width="100%"))

    def _help(self, text):
        return self.W.HTML(f'<p style="color:#496473;line-height:1.5;margin:12px 0">{text}</p>')

    def _build_setup(self):
        W = self.W
        self.root_field = W.Text(value=str(self.root or ""), placeholder="/home/you/whoi_cable")
        self.workspace_field = W.Text(value=str(self.workspace))
        self.setup_output = W.HTML()
        self.project_path = W.Text(placeholder="Path to project JSON or existing .cbl")
        self.project_list = W.Dropdown(options=[("No saved projects", "")])
        self.template_list = W.Dropdown(options=[("Blank surface mooring", "__blank__")]+[(p.stem,str(p)) for p in sorted((self.root / "cases").glob("*.cbl"))] if self.root else [("Blank surface mooring", "__blank__")])
        self.setup_tab = W.VBox([
            self._help("Use the <b>Python (whoi-cable)</b> kernel inside Linux/WSL. Point to the extracted repository containing <code>model/</code>, <code>cases/</code> and <code>scripts/</code>."),
            self._grid([self._label("WHOI Cable repository", self.root_field), self._label("Project and custom-parts folder", self.workspace_field)]),
            W.HBox([self._button("Apply paths", self._apply_paths, True), self._button("Check setup", self._diagnostics), self._button("Build both solvers", lambda: self._launch(self._build_solvers()))]), self.setup_output,
            self._help("<b>New project:</b> load one of the supplied cases or start a blank anchored mooring. Loading a project replaces the current form values; save your work first."),
            self._label("Starting configuration", self.template_list), self._button("Load template / new", self._template, True),
            self._grid([self._label("Saved projects", self.project_list), self._label("Or JSON / CBL path", self.project_path)]),
            W.HBox([self._button("Refresh list", self._refresh_project_list), self._button("Load project / CBL", self._load_project), self._button("Save project + CBL", self._save_project, True)]),
            self._help("Projects save their used parts alongside the layout, so they can be reopened with the same values. Custom parts are saved to <code>parts.json</code>. Exported <code>.cbl</code> files embed the used part definitions. Original databases and cases are untouched.")
        ])

    def _apply_paths(self):
        root = Path(self.root_field.value).expanduser().resolve() if self.root_field.value.strip() else None
        if root and not (root/"model/parser.y").is_file(): raise ValueError("Repository path does not contain model/parser.y")
        if not self.workspace_field.value.strip(): raise ValueError("Choose a project folder")
        workspace = Path(self.workspace_field.value).expanduser().resolve()
        # Keep unsaved in-memory parts when changing paths.
        previous = copy.deepcopy(self.catalog)
        current_parts = used_parts(self.project, previous)
        self.root, self.workspace = root, workspace
        self.catalog = load_catalog(root)
        self._overlay_catalog()
        for k in KINDS:
            for name, entry in previous[k].items():
                if entry.get("source", "").startswith("Custom") or name in current_parts[k]: self.catalog[k][name] = entry
        self.template_list.options = [("Blank surface mooring", "__blank__")] + ([(p.stem, str(p)) for p in sorted((root/"cases").glob("*.cbl"))] if root else [])
        self._refresh_choices()
        self._refresh_project_list()
        self._diagnostics()
        self._notice("Paths applied. Current mooring retained.")

    def _diagnostics(self):
        rows = [f"Python: {html.escape(sys.executable)}", f"Parts available: {sum(len(v) for v in self.catalog.values())}"]
        for program in ("cable-free", "cable", "res2mat"):
            try: location = resolve_program(self.root, program)
            except FileNotFoundError: location = "not found — build with the button above"
            rows.append(f"<b>{program}</b>: {html.escape(location)}")
        rows.append("Notebook dependencies: ipywidgets, NumPy, SciPy, Matplotlib. Add ipywidgets to the kernel; use jupyterlab_widgets in the environment that serves JupyterLab.")
        self.setup_output.value = '<div style="background:#f3f7f9;padding:12px;margin:10px 0">' + "<br>".join(rows) + "</div>"

    async def _build_solvers(self):
        if not self.root: raise ValueError("Set a WHOI Cable repository first")
        if os.name == "nt": raise ValueError("Build and run this CABLE version inside WSL/Linux; use a WSL Jupyter kernel")
        if os.environ.get("CONDA_DEFAULT_ENV") != "whoi-cable" or not os.environ.get("CONDA_PREFIX"):
            raise ValueError("Build requires the activated whoi-cable conda environment. Start Jupyter after conda activate whoi-cable, or use the existing scripts/build.sh in that environment.")
        self.tabs.selected_index = 4
        log = self.workspace / "build_gui.log"
        await self._process(["bash", str(self.root/"scripts/build.sh")], self.root, log)
        self._diagnostics()
        self._notice("Build completed. The solver paths are shown in Setup.")

    def _refresh_project_list(self):
        self.project_list.options = [(str(p.parent.name), str(p)) for p in sorted(self.workspace.glob("*/project.json"))] or [("No saved projects", "")]

    def _template(self):
        selected = self.template_list.value
        if selected == "__blank__": self.project = default_project()
        else:
            path = Path(selected)
            p, parts = import_cbl(path.read_text(), path.stem)
            merged = copy.deepcopy(self.catalog)
            for k in KINDS: merged[k].update(parts[k])
            validate_project(p, merged)
            self.project, self.catalog = p, merged
        self.raw_origin = None
        self.raw_mode.value = False
        self._load_widgets()
        self._notice("Configuration loaded. Edit the case name before saving a new variant.")

    def _load_project(self):
        filename = self.project_path.value.strip() or self.project_list.value
        if not filename: raise ValueError("Select a project or enter a file path")
        path = Path(filename).expanduser().resolve()
        if path.suffix.lower() == ".cbl":
            text = path.read_text(encoding="utf-8")
            try: p, parts = import_cbl(text, path.stem)
            except (ValueError, KeyError) as e:
                self.raw_editor.value, self.raw_origin = text, path.parent
                self.raw_mode.value = True
                self.tabs.selected_index = 5
                self._notice(f"Loaded in raw-input mode: {e}. Form values remain available; raw text is exported/run until you turn raw mode off.")
                return
        else:
            data = json.loads(path.read_text())
            if data.get("schema_version") != 1: raise ValueError("Unsupported project version")
            p, parts = data["project"], data["parts"]
        merged = copy.deepcopy(self.catalog)
        for k in KINDS: merged[k].update(parts.get(k, {}))
        validate_project(p, merged)
        self.project, self.catalog = p, merged
        self.raw_mode.value = False
        self.raw_origin = None
        self._load_widgets()
        self._notice(f"Loaded {path.name}, including its part definitions.")

    def _build_parts(self):
        W = self.W
        self.part_kind = W.Dropdown(options=[(label, k) for k,label in KINDS.items()])
        self.part_search = W.Text(placeholder="Search part names")
        self.part_list = W.Select(options=[], rows=12)
        self.part_name = W.Text()
        self.part_category = W.Text()
        self.part_comment = W.Text()
        self.part_source = W.HTML()
        self.part_form = W.VBox()
        self.part_extra = W.Textarea(placeholder="e.g. T = 1e5*e + 2e5*e*e", layout=W.Layout(height="90px"))
        self.part_preview = W.Output()
        self.part_kind.observe(self._guard(self._part_new), names="value")
        self.part_search.observe(self._guard(self._refresh_parts_list), names="value")
        self.part_list.observe(self._guard(self._part_select), names="value")
        self.parts_tab = W.VBox([
            self._help("Create a part or select an existing one to edit. <b>Lines:</b> mass kg/m and wet weight N/m. <b>Connectors:</b> total mass kg and wet weight N. Negative wet weight means uplift. Enter physical properties from a datasheet or measurements."),
            self._grid([self._label("Part family", self.part_kind), self._label("Search", self.part_search)]),
            W.HBox([W.VBox([self.part_list, self._button("New part", self._part_new), self._button("Duplicate selected", self._part_duplicate)], layout=W.Layout(width="28%", min_width="180px")),
                    W.VBox([self._grid([self._label("Part name", self.part_name), self._label("Category", self.part_category)]), self._label("Comment / provenance", self.part_comment), self.part_source, self.part_form,
                            self._label("Advanced properties (key = value, one per line)", self.part_extra),
                            W.HBox([self._button("Preview buoy geometry", self._preview_part), self._button("Save part", self._save_part, True)]), self.part_preview], layout=W.Layout(width="72%"))]),
            self._help("Connectors represent shackles, releases, sensors, downweights and subsurface floats. Use them between segments or attach them at segment nodes. A finite-length instrument with stiffness can instead be a Material segment. Buoy flotation is derived from the entered hull geometry and mass.")
        ])

    def _refresh_parts_list(self):
        names = sorted(self.catalog[self.part_kind.value])
        query = self.part_search.value.casefold()
        options = [n for n in names if query in n.casefold()]
        old = self.part_list.value
        self._loading = True
        self.part_list.options = options
        self.part_list.value = old if old in options else None
        self._loading = False

    def _part_new(self):
        self._refresh_parts_list()
        self._set_part_fields({k: copy.deepcopy(default) for k,_,default in FIELDS[self.part_kind.value] if default is not None})
        self.part_name.value = "new_part"
        self.part_category.value = "custom"
        self.part_comment.value = ""
        self.part_source.value = "<small>New part. Defaults are examples; replace them with your values.</small>"

    def _set_part_fields(self, props):
        W = self.W
        self._part_baseline = copy.deepcopy(props)
        self.part_inputs = {}
        widgets = []
        for key, label, default in FIELDS[self.part_kind.value]:
            value = props.get(key)
            if key == "type":
                options = ["linear", "nonlinear"] if self.part_kind.value == "Materials" else ["axisymmetric", "sphere", "cylinder"]
                field = W.Dropdown(options=options, value=value if value in options else options[0])
            elif key == "diameters":
                field = W.Textarea(value=rows_text(value) if value else "", layout=W.Layout(height="115px"))
            else: field = W.Text(value=fmt(value) if isinstance(value, (int,float)) else str(value) if value is not None else "", placeholder="blank = omit")
            self.part_inputs[key] = field
            widgets.append(self._label(label,field))
        self.part_form.children = (self._grid(widgets),)
        normal = {k for k,_,_ in FIELDS[self.part_kind.value]} | STRINGS
        self.part_extra.value = "\n".join(f"{k} = {v}" for k,v in props.items() if k not in normal)

    def _part_select(self):
        if self._loading or not self.part_list.value: return
        name = self.part_list.value
        entry = self.catalog[self.part_kind.value][name]
        props = entry["properties"]
        self._set_part_fields(props)
        self.part_name.value = name
        self.part_category.value = props.get("category", "")
        self.part_comment.value = props.get("comment", "")
        self.part_source.value = "<small>Source: " + html.escape(entry.get("source", "")) + "</small>"

    def _part_duplicate(self):
        if not self.part_list.value: raise ValueError("Select a part to duplicate")
        self._part_select()
        self.part_name.value += "_copy"
        self._notice("Duplicate ready. Change the name or properties and click Save part.")

    def _read_part_fields(self):
        kind = self.part_kind.value
        props = {}
        if self._part_baseline.get("color"): props["color"] = self._part_baseline["color"]
        for k,w in self.part_inputs.items():
            value = str(w.value).strip()
            if not value: continue
            if k == "diameters": props[k] = parse_rows(value)
            elif k == "type": props[k] = value
            else: props[k] = number(value)
        for line in self.part_extra.value.splitlines():
            if not line.strip(): continue
            if "=" not in line: raise ValueError("Advanced properties need key = value")
            k,v = [x.strip() for x in line.split("=",1)]
            if k not in EXTRA_KEYS[kind]: raise ValueError(f"Unsupported advanced property {k}")
            props[k] = v if k in {"T","Te","Tee"} else number(v)
        if self.part_category.value.strip(): props["category"] = self.part_category.value.strip()
        if self.part_comment.value.strip(): props["comment"] = self.part_comment.value.strip()
        # Unused shape inputs are omitted to avoid accidentally retaining old geometry.
        if kind == "Buoys":
            if props.get("type") != "axisymmetric": props.pop("diameters", None)
            if props.get("type") == "axisymmetric":
                props.pop("d", None)
                props.pop("h", None)
        validate_part(kind, self.part_name.value.strip(), props)
        return props

    def _save_part(self):
        name, kind = self.part_name.value.strip(), self.part_kind.value
        props = self._read_part_fields()
        catalog = copy.deepcopy(self.catalog)
        catalog[kind][name] = {"properties": props, "source": "Custom notebook part"}
        atomic_write(self.workspace / "parts.json", json.dumps({"schema_version":1,"parts":catalog}, indent=2, allow_nan=False))
        self.catalog = catalog
        self._refresh_choices()
        self._refresh_parts_list()
        self.part_list.value = name
        self._notice(f"Saved {name}. Moorings that reference this name now use these properties; saved project snapshots retain their earlier values.")

    def _preview_part(self):
        if self.part_kind.value != "Buoys": raise ValueError("Select Terminal buoy to preview geometry")
        props = self._read_part_fields()
        rho = number(self.env_fields["rho"].value)
        g = number(self.env_fields["gravity"].value)
        volume = buoy_volume(props)
        import numpy as np
        import matplotlib.pyplot as plt
        self.part_preview.clear_output(wait=True)
        with self.part_preview:
            if props["type"] == "axisymmetric": z,r = np.array(props["diameters"]).T; r=r/2
            elif props["type"] == "sphere":
                z=np.linspace(0,props["d"],160); r=np.sqrt(np.maximum(0,(props["d"]/2)**2-(z-props["d"]/2)**2))
            else: z=np.array([0,props["h"]]); r=np.full(2,props["d"]/2)
            fig, ax=plt.subplots(figsize=(4,4))
            ax.fill_betweenx(z,-r,r,color="#3ca6ba",alpha=.7)
            ax.set_aspect("equal"); ax.set_xlabel("Radius / width (m)"); ax.set_ylabel("Height above buoy base (m)"); ax.grid(alpha=.2)
            ax.set_title(self.part_name.value)
            self.display(fig); plt.close(fig)
            print(f"Hull volume: {1000*volume:.2f} L; geometric displacement: {rho*volume:.2f} kg")
            print(f"Fully immersed net uplift: {(rho*volume-props['m'])*g:.2f} N = {rho*volume-props['m']:.2f} kg equivalent")
            if props.get("buoyancy",0): print("Imported buoyancy override is present; figure above uses geometry.")

    def _build_mooring(self):
        W=self.W
        self.name_field=W.Text(); self.title_field=W.Text()
        self.type_field=W.Dropdown(options=[("Surface mooring","surface"),("Subsurface mooring","subsurface")])
        self.anchor_field=W.Dropdown(options=[]); self.buoy_field=W.Dropdown(options=[])
        self.segment_list=W.Select(options=[],rows=9)
        self.segment_material=W.Dropdown(options=[])
        self.segment_length=W.Text(value="50")
        self.segment_nodes=W.IntText(value=51)
        self.segment_connector=W.Dropdown(options=[("None","" )])
        self.attachments=W.Textarea(placeholder="part_name : 2, 5, 8\nshackle_10mm : 51",layout=W.Layout(height="85px"))
        self.layout_summary=W.HTML(); self.layout_preview=W.Output()
        self.segment_list.observe(self._guard(self._select_segment),names="value")
        self.mooring_tab=W.VBox([
            self._help("Build the line <b>from anchor upward</b>. Each row is a finite-length segment. Its connector is placed after that segment. Attachments sit at local nodes: node 1 is the lower end, node N the upper end. With uniform spacing, distance = (node − 1) × length / (N − 1)."),
            self._grid([self._label("Case name (file-safe)",self.name_field),self._label("Title",self.title_field),self._label("Problem type",self.type_field),self._label("Anchor",self.anchor_field),self._label("Terminal buoy",self.buoy_field)]),
            self._label("Ordered segments — bottom to top",self.segment_list),
            W.HBox([self._button("Add segment",self._add_segment,True),self._button("Update selected",self._update_segment),self._button("Copy selected",self._copy_segment),self._button("Move down / toward bed",lambda:self._move_segment(-1)),self._button("Move up / toward buoy",lambda:self._move_segment(1)),self._button("Remove selected",self._remove_segment)],layout=W.Layout(flex_flow="row wrap")),
            self._grid([self._label("Material",self.segment_material),self._label("Length (m)",self.segment_length),self._label("Number of nodes",self.segment_nodes),self._label("Connector after this segment",self.segment_connector)]),
            self._label("Node attachments — one part : node,node list per line; repeat a node to add multiple identical bodies",self.attachments),
            W.HBox([self._button("Validate + preview",self._preview_layout,True),self._button("Save project + CBL",self._save_project)]),self.layout_summary,self.layout_preview
        ])

    def _refresh_choices(self):
        def update(widget,options):
            old=widget.value
            widget.options=options
            vals=[v if not isinstance(v,tuple) else v[1] for v in options]
            if old in vals: widget.value=old
        for w,k in ((self.anchor_field,"Anchors"),(self.buoy_field,"Buoys"),(self.segment_material,"Materials")):
            update(w,sorted(self.catalog[k]))
        update(self.segment_connector,[("None","")]+[(n,n) for n in sorted(self.catalog["Connectors"])])

    def _refresh_segments(self, selected=None):
        opts=[]
        for i,s in enumerate(self.project["segments"]):
            extra=f" → {s['connector']}" if s.get("connector") else ""
            nattach=sum(len(a["nodes"]) for a in s.get("attachments",[]))
            if nattach: extra+=f" · {nattach} attachments"
            opts.append((f"{i+1:02d} | {s['material']} · {fmt(s['length'])} m · {s['nodes']} nodes{extra}",i))
        self._loading=True
        self.segment_list.options=opts
        self.segment_list.value=selected if selected is not None and selected<len(opts) else (0 if opts else None)
        self._loading=False
        self._select_segment()

    def _select_segment(self):
        if self._loading or self.segment_list.value is None: return
        s=self.project["segments"][self.segment_list.value]
        self.segment_material.value=s["material"]
        self.segment_length.value=fmt(s["length"])
        self.segment_nodes.value=integer(s["nodes"])
        self.segment_connector.value=s.get("connector") or ""
        self.attachments.value="\n".join(a["part"]+" : "+", ".join(str(n) for n in a["nodes"]) for a in s.get("attachments",[]))

    def _read_segment(self):
        s={"material":self.segment_material.value,"length":number(self.segment_length.value),"nodes":integer(self.segment_nodes.value),"connector":self.segment_connector.value or None,"attachments":[]}
        if s["length"]<=0 or s["nodes"]<2: raise ValueError("Segment length > 0 and nodes ≥ 2 are required")
        for line in self.attachments.value.splitlines():
            if not line.strip(): continue
            if ":" not in line: raise ValueError("Attachment format: part_name : node, node")
            name, nodes=line.rsplit(":",1)
            name=name.strip().strip('"')
            if name not in self.catalog["Connectors"]: raise ValueError(f"Unknown attachment {name}; create it as a Connector first")
            ns=[integer(n) for n in re.split(r"[,\s]+",nodes.strip())]
            if not ns or any(n<1 or n>s["nodes"] for n in ns): raise ValueError(f"Attachment node must lie in 1..{s['nodes']}")
            s["attachments"].append({"part":name,"nodes":ns})
        return s

    def _add_segment(self):
        s=self._read_segment()
        pos=self.segment_list.value+1 if self.segment_list.value is not None else len(self.project["segments"])
        self.project["segments"].insert(pos,s)
        self._refresh_segments(pos)
        self._notice("Segment added above the selected segment.")

    def _update_segment(self):
        i=self.segment_list.value
        if i is None: raise ValueError("Select a segment")
        self.project["segments"][i]=self._read_segment()
        self._refresh_segments(i)
        self._notice("Segment updated.")

    def _copy_segment(self):
        i=self.segment_list.value
        if i is None: raise ValueError("Select a segment")
        self.project["segments"].insert(i+1,copy.deepcopy(self.project["segments"][i]))
        self._refresh_segments(i+1)

    def _move_segment(self,delta):
        i=self.segment_list.value
        if i is None: raise ValueError("Select a segment")
        j=i+delta
        if 0<=j<len(self.project["segments"]):
            ss=self.project["segments"]; ss[i],ss[j]=ss[j],ss[i]
            self._refresh_segments(j)

    def _remove_segment(self):
        i=self.segment_list.value
        if i is None: raise ValueError("Select a segment")
        self.project["segments"].pop(i)
        self._refresh_segments(min(i,len(self.project["segments"])-1) if self.project["segments"] else None)

    def _build_forcing(self):
        W=self.W
        self.env_fields={}
        env_labels={"depth":"Water depth (m)","rho":"Water density (kg/m³)","gravity":"Gravity (m/s²)","x-wind":"Wind along x (m/s)","bottom-stiffness":"Bottom stiffness (N/m²)","bottom-damping":"Bottom damping (N s/m²)"}
        for k in env_labels:self.env_fields[k]=W.Text()
        self.current_field=W.Textarea(layout=W.Layout(height="115px"))
        self.modulation_field=W.Textarea(placeholder="Optional: (0,1) (30,2) (60,1), or an expression in t",layout=W.Layout(height="65px"))
        self.wave_h=W.Text();self.wave_t=W.Text();self.wave_phase=W.Text()
        self.analysis_fields={}
        labels={"duration":"Run duration (s)","ramp-time":"Wave ramp time (s)","time-step":"Solver time step (s)","static-relaxation":"Static relaxation", "static-iterations":"Static iterations","static-tolerance":"Static tolerance","static-outer-iterations":"Static outer iterations","static-outer-relaxation":"Static outer relaxation","static-outer-tolerance":"Static outer tolerance","dynamic-iterations":"Dynamic iterations","dynamic-relaxation":"Dynamic relaxation","dynamic-tolerance":"Dynamic tolerance","relax-adapt-up":"Relax adaptation up","relax-adapt-down":"Relax adaptation down","static-initial-guess":"Static initial guess","static-solution":"Static solution"}
        for k in labels:
            if k=="static-initial-guess":w=W.Dropdown(options=["catenary","shooting"])
            elif k=="static-solution":w=W.Dropdown(options=["relaxation","shooting","catenary"])
            else:w=W.Text()
            self.analysis_fields[k]=w
        advanced=W.Accordion(children=[self._grid([self._label(labels[k],self.analysis_fields[k]) for k in labels if k not in ("duration","ramp-time","time-step")])],selected_index=None)
        advanced.set_title(0,"Advanced static / dynamic solver controls")
        self.forcing_tab=W.VBox([
            self._help("This form uses a <b>2D x–vertical plane, regular waves and Morison forcing</b>. Surface-buoy submergence uses <code>cable-free</code>. Depth is positive downward from mean sea level; current is signed along x."),
            self._grid([self._label(label,self.env_fields[k]) for k,label in env_labels.items()]),
            self._label("Current profile: depth below mean surface (m), signed speed (m/s)",self.current_field),
            self._label("Optional current modulation: time (s), multiplier pairs — placed before x-current on export",self.modulation_field),
            self._grid([self._label("Regular wave height H (m)",self.wave_h),self._label("Wave period T (s)",self.wave_t),self._label("Wave phase (radians)",self.wave_phase)]),
            self._help("Height is crest to trough; the exporter writes CABLE amplitude H/2. H = 0 disables waves. The current modulation is a multiplier; leave it blank for steady forcing. Arbitrary expressions are checked by CABLE's parser."),
            self._grid([self._label(labels[k],self.analysis_fields[k]) for k in ("duration","ramp-time","time-step")]),advanced,
            self._button("Validate settings",self._validate,True)
        ])

    def _collect(self):
        p=copy.deepcopy(self.project)
        p.update(name=self.name_field.value.strip(),title=self.title_field.value.strip(),type=self.type_field.value,anchor=self.anchor_field.value,buoy=self.buoy_field.value)
        for k,w in self.env_fields.items():p["environment"][k]=number(w.value)
        for k,w in self.analysis_fields.items():p["analysis"][k]=w.value if k in ("static-initial-guess","static-solution") else number(w.value)
        p["current"]=parse_rows(self.current_field.value)
        p["wave"]=[number(w.value) for w in (self.wave_h,self.wave_t,self.wave_phase)]
        p["modulation"]=self.modulation_field.value.strip()
        p["run"]={"solver":self.solver_field.value,"static_only":self.static_field.value,"auto":self.auto_field.value,"sample":number(self.sample_field.value),"snap_dt":number(self.snap_field.value)}
        # A displayed segment is an edit buffer. Save/preview/run commits it automatically.
        if self.segment_list.value is not None:p["segments"][self.segment_list.value]=self._read_segment()
        validate_project(p,self.catalog)
        self.project=p
        return p

    def _load_widgets(self):
        p=self.project
        self._refresh_choices()
        self.name_field.value=p["name"];self.title_field.value=p["title"];self.type_field.value=p["type"]
        self.anchor_field.value=p["anchor"];self.buoy_field.value=p["buoy"]
        for k,w in self.env_fields.items():w.value=fmt(p["environment"][k])
        for k,w in self.analysis_fields.items():w.value=str(p["analysis"][k])
        self.current_field.value=rows_text(p["current"])
        self.modulation_field.value=p.get("modulation","")
        for w,v in zip((self.wave_h,self.wave_t,self.wave_phase),p["wave"]):w.value=fmt(v)
        r=p["run"];self.solver_field.value=r["solver"];self.static_field.value=r["static_only"];self.auto_field.value=r["auto"]
        self.sample_field.value=fmt(r["sample"]);self.snap_field.value=fmt(r["snap_dt"])
        self._refresh_segments()
        self._refresh_parts_list()

    def _validate(self):
        p=self._collect()
        warnings=validate_project(p,self.catalog)
        self._notice("Valid configuration."+(" " + " ".join(warnings) if warnings else " Use Check CABLE syntax before running."))

    def _preview_layout(self):
        p=self._collect()
        warnings=validate_project(p,self.catalog)
        total=sum(s["length"] for s in p["segments"])
        nodes=sum(s["nodes"] for s in p["segments"])
        wet=sum(s["length"]*number(self.catalog["Materials"][s["material"]]["properties"]["wet"]) for s in p["segments"])
        for s in p["segments"]:
            if s.get("connector"):wet+=number(self.catalog["Connectors"][s["connector"]]["properties"].get("wet",0))
            wet+=sum(len(a["nodes"])*number(self.catalog["Connectors"][a["part"]]["properties"]["wet"]) for a in s.get("attachments",[]))
        self.layout_summary.value=f'<div style="padding:12px;background:#eaf4f7"><b>{total:.2f} m</b> line · scope {total/p["environment"]["depth"]:.2f} · {nodes} segment-local nodes · net line/hardware wet weight {wet:.1f} N<br><small>Preview is an assembly diagram, not an equilibrium calculation. Node count sums segment-local counts.</small></div>'
        import matplotlib.pyplot as plt
        self.layout_preview.clear_output(wait=True)
        with self.layout_preview:
            fig,ax=plt.subplots(figsize=(8,5))
            colors=["#2488a5","#efaa43","#7861a9","#46986b"]
            y=0
            for i,s in enumerate(p["segments"]):
                end=y+s["length"]
                ax.plot([0,0],[y,end],lw=7,color=colors[i%len(colors)],solid_capstyle="butt")
                ax.text(.12,(y+end)/2,f"{i+1}. {s['material']}\n{s['length']:g} m, {s['nodes']} nodes",va="center",fontsize=9)
                for a in s.get("attachments",[]):
                    for n in a["nodes"]:
                        yp=y+(n-1)*s["length"]/(s["nodes"]-1)
                        ax.plot(-.05,yp,"o",color="#d05a48",ms=5)
                    ax.text(-.1,y+(a["nodes"][0]-1)*s["length"]/(s["nodes"]-1),a["part"],ha="right",fontsize=8)
                y=end
                if s.get("connector"):
                    ax.plot(0,y,"s",color="#10354b",ms=7)
                    ax.text(-.1,y,s["connector"],ha="right",fontsize=8)
            ax.plot(0,0,"v",color="#10354b",ms=12);ax.plot(0,total,"o",color="#3ca6ba",ms=15)
            ax.text(.12,0,p["anchor"],va="top");ax.text(.12,total,p["buoy"],va="bottom")
            ax.set_xlim(-1.1,1.1);ax.set_ylim(-total*.08,total*1.12)
            ax.set_xticks([]);ax.set_ylabel("Distance along unstretched assembly from anchor (m)")
            ax.spines[['top','right','bottom']].set_visible(False)
            ax.set_title("Mooring assembly — anchor to terminal buoy")
            fig.tight_layout();self.display(fig);plt.close(fig)
        self._notice("Assembly preview updated."+(" " + " ".join(warnings) if warnings else ""))

    def _save_project(self):
        p=self._collect()
        folder=self.workspace/p["name"]
        data={"schema_version":1,"designer_version":VERSION,"project":p,"parts":used_parts(p,self.catalog)}
        text=export_cbl(p,self.catalog)
        atomic_write(folder/"project.json",json.dumps(data,indent=2,allow_nan=False))
        atomic_write(folder/(p["name"]+".cbl"),text)
        self._refresh_project_list()
        self._notice(f"Saved project.json and {p['name']}.cbl in {folder}."+(" Raw-input edits are separate; use Export active input for those." if self.raw_mode.value else ""))

    def _build_input(self):
        W=self.W
        self.raw_mode=W.Checkbox(value=False,description="Use raw input for export / syntax check / solver run",indent=False)
        self.raw_editor=W.Textarea(placeholder="Click Generate from forms, or load an existing .cbl in Setup",layout=W.Layout(height="480px",width="100%"))
        self.raw_editor.add_class("cable-source")
        self.input_tab=W.VBox([
            self._help("Review generated input or edit CABLE syntax directly. <b>Raw mode runs the text below;</b> form edits do not update it automatically. Turn raw mode off to return to the form configuration. Unsupported imported layouts are kept here without rewriting them."),
            self.raw_mode,W.HBox([self._button("Generate from forms",self._generate_input,True),self._button("Export active input",self._export_input)]),self.raw_editor,
            self._help("Raw mode uses <code>-nocpp</code> and preserves CABLE input text. Imported relative forcing-file paths are resolved from the original CBL directory. Add an absolute path for external files when moving a case. Raw text is checked by the installed solver; the Python forms do not validate it.")
        ])

    def _generate_input(self):
        self.raw_editor.value=export_cbl(self._collect(),self.catalog)
        self.raw_origin=None
        self._notice("Input generated from forms. Tick raw mode only if you want your text edits to control export/runs.")

    def _active_input(self):
        if self.raw_mode.value:
            text=self.raw_editor.value
            if not text.strip():raise ValueError("Raw input is empty")
            name=self.name_field.value.strip()
            if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_-]*",name):raise ValueError("Choose a file-safe case name on the Mooring tab")
            run={"solver":self.solver_field.value,"static_only":self.static_field.value,"auto":self.auto_field.value,"sample":number(self.sample_field.value),"snap_dt":number(self.snap_field.value)}
            if min(run["sample"],run["snap_dt"])<=0:raise ValueError("Output intervals must be positive")
            return name,text,run
        p=self._collect()
        return p["name"],export_cbl(p,self.catalog),p["run"]

    def _export_input(self):
        name,text,_=self._active_input()
        dest=self.workspace/name/(name+"_raw.cbl" if self.raw_mode.value else name+".cbl")
        atomic_write(dest,text)
        self._notice(f"Exported {dest}")

    def _build_run(self):
        W=self.W
        self.solver_field=W.Dropdown(options=[("cable-free (free surface buoy)","cable-free"),("cable (stock)","cable")])
        self.solver_path=W.Text(placeholder="Optional executable override")
        self.converter_path=W.Text(placeholder="Optional res2mat executable override")
        self.static_field=W.Checkbox(description="Static equilibrium only",indent=False)
        self.auto_field=W.Checkbox(description="Use -auto static strategy",indent=False)
        self.sample_field=W.Text(value="0.1");self.snap_field=W.Text(value="0.2")
        self.timeout_field=W.IntText(value=3600)
        self.run_log=W.Textarea(value="Ready to check or run.\n",disabled=True,layout=W.Layout(height="240px",width="100%"))
        self.results=W.Output()
        self.result_path=W.Text(placeholder="Optional path to an existing .mat result")
        self.run_buttons=[self._button("Check CABLE syntax",lambda:self._launch(self._run(check=True))),self._button("Run solver",lambda:self._launch(self._run()),True)]
        self.stop_button=self._button("Stop",self._stop)
        self.stop_button.disabled=True
        self.run_tab=W.VBox([
            self._help("Runs start from a static solution. <code>cable-free</code> uses the supplied surface-buoy patch; its buoy-patch behavior applies to 2D surface moorings with Morison forcing. A syntax check does not solve equilibrium or test convergence."),
            self._grid([self._label("Solver",self.solver_field),self._label("Executable path (blank = discover)",self.solver_path),self._label("res2mat path (blank = discover)",self.converter_path)]),
            W.HBox([self.static_field,self.auto_field]),
            self._grid([self._label("Time-history output interval (s)",self.sample_field),self._label("Whole-line snapshot interval (s)",self.snap_field),self._label("Timeout per process (s)",self.timeout_field)]),
            W.HBox(self.run_buttons+[self.stop_button]),self.run_log,
            self._help("Each run has a fresh timestamped folder under <code>runs/</code>, an exact copy of the active input, the command, a log and solver output. Conversion produces a <code>.mat</code> file for this panel and the supplied <code>plot_run.ipynb</code>. Use small time steps and inspect convergence messages before interpreting results."),
            self._label("Existing result MAT file (leave blank for latest run)",self.result_path),self._button("Plot result",self._plot_results),self.results
        ])

    def _launch(self,coro):
        if self._busy:
            coro.close();raise ValueError("A process is already running; stop it or wait")
        try:loop=asyncio.get_running_loop()
        except RuntimeError:
            coro.close();raise RuntimeError("Launch this app inside a running Jupyter kernel")
        self._busy=True;self.cancel_requested=False
        for b in self.run_buttons:b.disabled=True
        self.stop_button.disabled=False
        self.run_log.value=""
        async def wrapped():
            try:await coro
            except asyncio.CancelledError:self._notice("Process stopped. Partial output is retained.")
            except asyncio.TimeoutError:self._notice("Process exceeded the timeout and was stopped. Partial output and the log are retained.",True)
            except Exception as e:self._notice(f"{type(e).__name__}: {e}",True)
            finally:
                self._busy=False;self.process=None
                for b in self.run_buttons:b.disabled=False
                self.stop_button.disabled=True
        self.task=loop.create_task(wrapped())

    def _stop(self):
        self.cancel_requested=True
        if self.process and self.process.returncode is None:
            self._signal_process(self.process)
        self._notice("Stopping process; output produced so far will be retained.")

    @staticmethod
    def _signal_process(process,kill=False):
        try:
            if os.name != "nt":os.killpg(process.pid,signal.SIGKILL if kill else signal.SIGTERM)
            elif kill:process.kill()
            else:process.terminate()
        except ProcessLookupError:pass

    async def _process(self,command,cwd,log):
        timeout=integer(self.timeout_field.value)
        if timeout<=0:raise ValueError("Timeout must be positive")
        log=Path(log);log.parent.mkdir(parents=True,exist_ok=True)
        self.run_log.value += "$ " + shlex.join(command) + "\n"
        self.process=await asyncio.create_subprocess_exec(*command,cwd=str(cwd),stdout=asyncio.subprocess.PIPE,stderr=asyncio.subprocess.STDOUT,start_new_session=(os.name != "nt"))
        process=self.process
        if self.cancel_requested:self._signal_process(process)
        async def consume():
            with log.open("w",encoding="utf-8") as out:
                while True:
                    chunk=await process.stdout.read(16384)
                    if not chunk:break
                    message=chunk.decode("utf-8",errors="replace")
                    out.write(message);out.flush()
                    # Preserve the full disk log; bound only the live notebook display.
                    self.run_log.value=(self.run_log.value+message)[-28000:]
            return await process.wait()
        try:rc=await asyncio.wait_for(consume(),timeout)
        except (asyncio.TimeoutError,asyncio.CancelledError):
            if process.returncode is None:
                self._signal_process(process)
                try:await asyncio.wait_for(process.wait(),3)
                except asyncio.TimeoutError:self._signal_process(process,kill=True);await process.wait()
            raise
        if self.cancel_requested:raise asyncio.CancelledError()
        if rc:raise RuntimeError(f"Process exited with status {rc}; see {log}")
        return log.read_text(encoding="utf-8",errors="replace")

    async def _run(self,check=False):
        name,text,run=self._active_input()
        root=self.root
        executable=resolve_program(root,run["solver"],self.solver_path.value)
        stamp=dt.datetime.now().strftime("%Y%m%d_%H%M%S_%f")
        folder=(self.workspace/"checks" if check else (self.root/"runs" if self.root else self.workspace/"runs"))/(name+"_"+stamp)
        folder.mkdir(parents=True,exist_ok=False)
        cbl=folder/(name+".cbl");cab=folder/(name+".cab")
        cbl.write_text(text,encoding="utf-8")
        if not self.raw_mode.value:
            atomic_write(folder/"project.json",json.dumps({"schema_version":1,"designer_version":VERSION,"project":self.project,"parts":used_parts(self.project,self.catalog)},indent=2,allow_nan=False))
        command=solver_command(executable,cbl,cab,run,check)
        cwd=self.raw_origin or folder
        atomic_write(folder/"run.json",json.dumps({"command":command,"cwd":str(cwd),"raw_mode":self.raw_mode.value,"started":dt.datetime.now().isoformat()},indent=2))
        self._notice("Checking input syntax…" if check else f"Running {run['solver']}… Stop remains available.")
        output=await self._process(command,cwd,folder/(name+".log"))
        if re.search(r"syntax error|parse error|undefined|could not read|error in",output,re.I):raise RuntimeError(f"CABLE reported an input error; see {folder}")
        if check:
            self._notice(f"CABLE parsed the input. Check log: {folder}")
            return
        if not cab.is_file() or not cab.stat().st_size:raise RuntimeError("Solver produced no output file; inspect its log")
        self.last_run={"folder":folder,"name":name,"mat":None,"static_only":run["static_only"]}
        warnings=bool(re.search(r"never converged|could not get static|did not converge|exceeded|\bnan\b|diverge|failed to converge",output,re.I))
        try:converter=resolve_program(root,"res2mat",self.converter_path.value)
        except FileNotFoundError:
            self._notice(f"Solver finished. res2mat is missing; .cab and log are in {folder}",True)
            return
        mat=folder/(name+".mat")
        await self._process([converter,"-in",str(cab),"-out",str(mat),"-totals"],folder,folder/"convert.log")
        from scipy.io import loadmat
        import numpy as np
        result=loadmat(mat)
        if any(not np.isfinite(result[k]).all() for k in ("x","z","T","x_t","z_t","T_t") if k in result):warnings=True
        if not run["static_only"]:
            duration=assignments(sections(text).get("analysis parameters","")).get("duration")
            times=np.asarray(result.get("t",[])).ravel()
            tolerance=2*float(np.asarray(result.get("dt",[[0.1]])).ravel()[0])
            if duration and (not len(times) or times[-1]<number(duration)-tolerance):warnings=True
        self.last_run["mat"]=mat
        self.result_path.value=str(mat)
        self._plot_results()
        self._notice(f"Run finished. Results: {folder}"+(". Convergence, incomplete-output or nonfinite-result warnings are present; inspect the log and output before using the result." if warnings else ". Inspect the log and test sensitivity to the time step."),warnings)

    def _plot_results(self):
        path=Path(self.result_path.value).expanduser() if self.result_path.value.strip() else self.last_run.get("mat") if self.last_run else None
        if not path:raise ValueError("Run a case or enter the path to a .mat result")
        from scipy.io import loadmat
        import numpy as np
        import matplotlib.pyplot as plt
        R=loadmat(path,squeeze_me=False)
        H=float(np.asarray(R.get("depth",[[self.project["environment"]["depth"]]])).ravel()[0])
        x=np.asarray(R["x"]).ravel();z=np.asarray(R["z"]).ravel()-H
        self.results.clear_output(wait=True)
        with self.results:
            print("Result:",path)
            fig,axes=plt.subplots(1,2,figsize=(11,4.5))
            ax=axes[0]
            ax.plot(x,z,color="#125970",lw=2,label="Static")
            if "x_s" in R and "z_s" in R:
                xs,zs=np.asarray(R["x_s"]),np.asarray(R["z_s"])-H
                if xs.ndim==2:
                    for k in np.unique(np.linspace(0,xs.shape[1]-1,min(20,xs.shape[1])).astype(int)):
                        ax.plot(xs[:,k],zs[:,k],color="#91aab4",alpha=.35,lw=.7)
                    ax.plot(xs[:,-1],zs[:,-1],color="#e19338",lw=1.5,label="Last snapshot")
            ax.axhline(0,color="#62a8c3",ls="--");ax.axhline(-H,color="#81766b")
            ax.set_xlabel("Horizontal distance (m)");ax.set_ylabel("z relative to mean sea level (m)")
            ax.set_title("Mooring geometry");ax.legend();ax.grid(alpha=.2)
            ax=axes[1]
            if "T" in R:
                tension=np.asarray(R["T"]).ravel()/1000
                s=np.asarray(R.get("s",np.arange(len(tension)))).ravel()
                ax.plot(tension,s,color="#125970")
                ax.set_xlabel("Static tension (kN)");ax.set_ylabel("Distance along cable (m)")
                ax.set_title("Static tension");ax.grid(alpha=.2)
            else:ax.axis("off")
            fig.tight_layout();self.display(fig);plt.close(fig)
            if "t" in R and "z_t" in R and np.asarray(R["t"]).size>1:
                t=np.asarray(R["t"]).ravel();zt=np.asarray(R["z_t"])
                fig,axes=plt.subplots(2,1,figsize=(10,5),sharex=True)
                if zt.shape[0]!=len(t) and zt.shape[1]==len(t):zt=zt.T
                axes[0].plot(t,zt[:,-1]-H,color="#125970");axes[0].set_ylabel("Terminal base z (m)")
                if "T_t" in R:
                    tt=np.asarray(R["T_t"])
                    if tt.shape[0]!=len(t) and tt.shape[1]==len(t):tt=tt.T
                    axes[1].plot(t,np.nanmax(tt,axis=1)/1000,color="#e19338")
                    axes[1].set_ylabel("Max sampled tension (kN)")
                axes[1].set_xlabel("Time (s)")
                for ax in axes:ax.grid(alpha=.2)
                fig.tight_layout();self.display(fig);plt.close(fig)
                duration=assignments(sections(path.with_suffix('.cbl').read_text()).get("analysis parameters","")).get("duration") if path.with_suffix('.cbl').is_file() else None
                if duration and t[-1] < number(duration)-2*float(np.asarray(R.get("dt",[[0.1]])).ravel()[0]):
                    print(f"WARNING: output ends at {t[-1]:g} s, earlier than requested {duration} s.")
            print("For animations, open the supplied notebooks/plot_run.ipynb and set RUN_DIR to",path.parent)


In [6]:
from IPython.display import display

gui = CableDesigner(cable_root=CABLE_ROOT, workspace=PROJECTS_DIR)
display(gui.widget)


## Using the interface

1. **Setup & projects:** choose repository/project folders and load a template, saved project or `.cbl` file. **Build both solvers** runs the attachment's build script if this Jupyter session inherits the activated `whoi-cable` environment. You can also use already-built binaries.
2. **Parts:** choose Materials, Connectors, Buoys or Anchors. Click **New part** or select and duplicate an existing part; enter a unique name and properties, then **Save part**. Geometry preview reports hull volume and net uplift. Built-in properties are starting points; verify them against your own equipment.
3. **Mooring:** choose the anchor and terminal buoy. Add, edit, copy, reorder or remove segments in anchor-to-buoy order. Choose connectors between segments. For attached instruments/floats, enter `part_name : node, node` lines. Click **Update selected** before switching to another segment; save/preview/run also commits the currently displayed edit buffer.
4. **Forcing & solver:** set depth, density, current profile, regular wave height/period, run duration and time step. Current depth is positive down from mean sea level. Wave height is converted to amplitude H/2 on export. More solver controls are in the accordion.
5. **CABLE input:** generate/review the input; optionally tick raw mode and edit CABLE text. In raw mode this text controls export and runs; generating from forms replaces it. Unsupported imported syntax goes here without modifying the form project.
6. **Run & results:** check input syntax, choose static/dynamic and stock/free solver, then run. Stop and timeout retain partial files. Plot converted results here; use the supplied `plot_run.ipynb` for animations.

**Saving:** `notebook_projects/parts.json` holds saved part overrides. Each project directory holds `project.json` plus an embedded-parts `.cbl` file. Saved JSON projects include snapshots of the parts they use. Solver runs create fresh timestamped folders under `<repository>/runs/` (or the project folder if no repository is set). The original cases and databases are not edited.

**Units:** line mass kg/m; line wet weight N/m; connector/anchor/buoy mass kg; connector/anchor wet weight N. Negative wet weight means uplift. Stiffness EA is N; EI and GJ are N m²; dimensions are metres. Buoy mass is not its displacement or rated net uplift.

**Notes:** the layout preview shows the unstretched assembly, not equilibrium. The CABLE parser check does not establish physical validity or convergence. The GUI runs the existing compiled C solvers, not a Python substitute. Long raw or form runs remain responsive via asynchronous subprocesses. Imported relative external-file paths use the input's original directory; preprocessors and `#include` are disabled (`-nocpp`). See `docs/NOTEBOOK_GUI.md` for installation and the full workflow.
